# Update Test Iteration

This notebook gets the most recent results from **Test Monitor** and checks for the **Test Iteration Property**
- It skips all the results with **Test Iteration Property**
- For results without one, it checks historically for older results with **Test Iteration**
- When there's no older **Test Iterations** it assigns its first Iteration **(Iteration 0)**
- When there's an older **Test Iteration** it assigns its consecutive Iteration

### Imports
Import Python modules for executing the notebook. Pandas is used for building and handling dataframes. Scrapbook is used for recording data for the Notebook Execution Service. The SystemLink Test Monitor Client provides access to test result data for processing.

In [ ]:
import pandas as pd
import scrapbook as sb
import time
import datetime
import copy
import numpy as np
from sys import exit

import systemlink.clients.nitestmonitor as testmon
import systemlink.clients.nitestmonitor.models as tm_models

### Parameters
- `results_filter`: Number of hours the automation should look for new results (DD.HH:mm:ss)
  Default: `30 min`
- `iteration_filter`: Set the conditions to create the Test Iteration property: properties ->"Test Iteration" is empty or doesn't exist, Status Type is not "running", and serialNumber is not empty  

In [ ]:
results_filter = 'startedWithin <= "0.0:30:0"'
iteration_filter = 'string.IsNullOrEmpty(properties[\"Test Iteration\"]) AND status.statusType != \"Running\" AND !string.IsNullOrEmpty(serialNumber)'

### Create Test Monitor client
Establish a connection to SystemLink over HTTP.

In [ ]:
api_client = testmon.ApiClient()
results_api = testmon.ResultsApi(api_client)
steps_api = testmon.StepsApi(api_client)

### Query for results
Query the Test Monitor Service for results matching the `results_filter` parameter. Then query for all test steps in those results.

In [ ]:
#Define batched query function
async def perform_batched_query(query_function, query, response_field):
    results = []
    response = await query_function(post_body=query)
    while response.continuation_token:
        results = results + getattr(response, response_field)
        query.continuation_token = response.continuation_token
        response = await query_function(post_body=query)
    return results

In [ ]:
results_filter = iteration_filter + ' AND ' + results_filter
results_query = testmon.ResultsAdvancedQuery(
    results_filter, order_by=testmon.ResultField.STARTED_AT
)
results = await perform_batched_query(results_api.query_results_v2, results_query, 'results')
results_list = []
if (len(results) != 0):
    temp = []
    for result in results:

        result_dict = result.to_dict()
        temp_val = result_dict['serial_number']
        if '"' not in temp_val: 
            if temp_val not in temp:
                temp.append(temp_val)
                results_list.append(result_dict)

#print(len(results))
#results


## Second Query
 Creates a Query for each serial number in the first list, to call for all the previous iterations

In [ ]:
#Second Query (Matt)

rcontainer = []
results_list2 =[]

#--- To run in chunks, uncomment these two lines and adjust the size in brackets. ---
#print(len(results_list))
#results_list = results_list[:1000]

for result in results_list:
    if result["serial_number"]:
        results_filter2 = 'serialNumber = "' + result["serial_number"] + '"'
        results_query2 = testmon.ResultsAdvancedQuery(
            results_filter2, order_by=testmon.ResultField.STARTED_AT, take=1000
        )
        results2 = await perform_batched_query(results_api.query_results_v2, results_query2, 'results')
    
        rcontainer = rcontainer + results2

# Convert to dict AFTER the loop, not inside it
results_list2 = [result.to_dict() for result in rcontainer]

group_names = []
for results in results2:
    if grouping in results2:
        group_names.append(results2[grouping])
#print(results_list2)


## Create Pandas DataFrame

In [ ]:
formatted_results = {
    'serial_number': [results2['serial_number'] for results2 in results_list2],
    'started_at': [results2['started_at'] for results2 in results_list2],
    'id': [results2['id'] for results2 in results_list2],
    'program_name': [results2['program_name'] for results2 in results_list2],
    'properties': [results2['properties'] for results2 in results_list2]
}


df_results = pd.DataFrame.from_dict(formatted_results)        
sorting_list = ['serial_number', 'started_at']
grouping_list = ['serial_number', 'program_name']
df_results = df_results.drop_duplicates(subset=['id'])

# Sorting...

 Sorts the data and assigns the Iteration number to each Result ID

In [ ]:
df_results = df_results.sort_values(by=['started_at', 'serial_number']).reset_index()
del df_results["index"]
df_results['Iteration'] =  df_results.groupby(['serial_number', 'program_name']).cumcount()
#df_results

# Update Test Results

Updates the Test Result with a property containing the Test Iteration



In [ ]:
new_result = []
for ind in df_results.index:
    test_props = {}
    result_id = str(df_results['id'][ind])
    test_props["Test Iteration"] = str(df_results['Iteration'][ind])
    new_result.append(tm_models.test_result_update_request_object.TestResultUpdateRequestObject(id=result_id, properties=test_props))

# Single API call AFTER building the complete list
request_body = tm_models.update_test_results_request.UpdateTestResultsRequest(results=new_result, replace=False)
ret_val = await results_api.update_results_v2(request_body)
#request_body